In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

print("DuckDB connected to Hugging Face ✅")

DuckDB connected to Hugging Face ✅


In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"

daily_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

print(daily_path)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet


In [4]:
test_df = con.sql(f"""
    SELECT *
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE month = '2026-03'
    LIMIT 5
""").df()

test_df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
print(test_df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [6]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS distinct_grain_rows
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE month = '2026-03'
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows
0,9841378,9841378


In [7]:
date_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE month = '2026-03'
""").df()

date_check

,row_count,min_report_date,max_report_date,distinct_dates
0,9841378,2026-03-01,2026-03-31,31


In [8]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_unavailable_rows
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE month = '2026-03'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,gsc_unavailable_rows
0,9841378,3611061,6230317


In [9]:
feature_check = con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) AS impressions_non_null,
        COUNT(*) FILTER (WHERE gsc_clicks IS NOT NULL) AS clicks_non_null,
        COUNT(*) FILTER (WHERE gsc_avg_position IS NOT NULL) AS avg_position_non_null,
        COUNT(*) FILTER (WHERE ga4_pageviews IS NOT NULL) AS pageviews_non_null,
        COUNT(*) FILTER (WHERE ga4_sessions IS NOT NULL) AS sessions_non_null
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE month = '2026-03'
""").df()

feature_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_available_rows,impressions_non_null,clicks_non_null,avg_position_non_null,pageviews_non_null,sessions_non_null
0,3611061,9841378,9841378,3611061,6822637,6822637


In [10]:
feature_availability = con.sql(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_impressions IS NOT NULL
        ) AS impressions_available,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_clicks IS NOT NULL
        ) AS clicks_available,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_avg_position IS NOT NULL
        ) AS avg_position_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
              AND ga4_pageviews IS NOT NULL
        ) AS pageviews_available,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
              AND ga4_sessions IS NOT NULL
        ) AS sessions_available
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE month = '2026-03'
""").df()

feature_availability

,impressions_available,clicks_available,avg_position_available,pageviews_available,sessions_available
0,3611061,3611061,3611061,413966,413966


In [11]:
future_check = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS total_dates
    FROM read_parquet('{daily_path}', hive_partitioning=true)
""").df()

future_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,total_dates
0,2025-01-27,2026-06-30,520


In [12]:
future_label_check = con.sql(f"""
    SELECT
        COUNT(DISTINCT content_hash_id) AS march_contents,
        COUNT(DISTINCT CASE
            WHEN report_date >= '2026-04-01'
             AND report_date <= '2026-04-30'
            THEN content_hash_id
        END) AS april_contents
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE report_date >= '2026-03-01'
      AND report_date <= '2026-04-30'
""").df()

future_label_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_contents,april_contents
0,362173,362172


In [13]:
label_check = con.sql(f"""
    SELECT
        COUNT(*) AS april_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND gsc_clicks IS NOT NULL
        ) AS april_gsc_rows
    FROM read_parquet('{daily_path}', hive_partitioning=true)
    WHERE report_date >= '2026-04-01'
      AND report_date <= '2026-04-30'
""").df()

label_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,april_rows,april_gsc_rows
0,10424730,3901060


In [14]:
leakage_df = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS march_clicks
        FROM read_parquet('{daily_path}', hive_partitioning=true)
        WHERE report_date >= '2026-03-01'
          AND report_date <= '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_clicks) AS future_april_clicks
        FROM read_parquet('{daily_path}', hive_partitioning=true)
        WHERE report_date >= '2026-04-01'
          AND report_date <= '2026-04-30'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.march_clicks,
        a.future_april_clicks
    FROM march m
    INNER JOIN april a
        ON m.client_hash_id = a.client_hash_id
       AND m.content_hash_id = a.content_hash_id
""").df()

print("Rows:", len(leakage_df))
leakage_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 158549


,client_hash_id,content_hash_id,march_clicks,future_april_clicks
0,client_73cda7b4e4f265ea,content_287262a30125d9e0,2.0,0.0
1,client_73cda7b4e4f265ea,content_653b0e6a2f03d727,0.0,0.0
2,client_73cda7b4e4f265ea,content_bfd0d577371d456e,3.0,4.0
3,client_73cda7b4e4f265ea,content_941dee84048d958a,0.0,0.0
4,client_73cda7b4e4f265ea,content_f2c9500b9fed088a,2.0,0.0


In [15]:
from scipy.stats import spearmanr

honest_corr, _ = spearmanr(
    leakage_df["march_clicks"],
    leakage_df["future_april_clicks"]
)

leaked_corr, _ = spearmanr(
    leakage_df["future_april_clicks"],
    leakage_df["future_april_clicks"]
)

print(f"Honest feature correlation : {honest_corr:.4f}")
print(f"Leaked feature correlation : {leaked_corr:.4f}")

Honest feature correlation : 0.7314
Leaked feature correlation : 1.0000


## ML-04 Data Contract Summary
### 1. Data Contract

- **Row meaning:** One row represents one content item for one pseudonymized client on one report date.
- **Table used:** `fact_content_daily_performance`
- **Development window:** March 2026 (`2026-03-01` to `2026-03-31`)
- **Ranking target:** Rank content opportunities for review using historical search and engagement signals. Future-period performance is used only as an evaluation proxy.
- **Deliberately excluded:** Future-period performance metrics, because they are not available at decision time.

### 2. Verified Data Facts

- March 2026 contains **9,841,378 rows**.
- Date range is **2026-03-01 to 2026-03-31**, with **31 distinct dates**.
- Grain check confirmed that `report_date + client_hash_id + content_hash_id` uniquely identifies each row.
- `gsc_data_available IS TRUE` identified **3,611,061** GSC-available rows.

### 3. Decision-Time Features

1. `gsc_impressions` — available when `gsc_data_available IS TRUE`.
2. `gsc_clicks` — available when `gsc_data_available IS TRUE`.
3. `gsc_avg_position` — available when `gsc_data_available IS TRUE`.
4. `ga4_pageviews` — available when `ga4_data_available IS TRUE`.
5. `ga4_sessions` — available when `ga4_data_available IS TRUE`.

### 4. Leakage Trap

A deliberately leaked feature, `future_april_clicks`, was created from the April 2026 outcome.

- Honest `march_clicks` Spearman correlation: **0.7314**
- Leaked `future_april_clicks` Spearman correlation: **1.0000**

The leaked feature was removed because it directly contains future outcome information.

### 5. Limitation

The warehouse is an unbalanced panel, so historical coverage and data availability differ across clients and content items. Rankings should therefore be interpreted as prioritization signals within the observed data, not as universal causal conclusions.
